# 11. 수치 오류 발생 단계 귀속

기존 10번 관계형 평가에서 틀린 수치만 가져와 다음 세 값을 연결한다.

`수기 gold → 저장된 OCR 페이지 텍스트 → 저장된 예측 JSON`

API를 다시 호출하지 않는다. 각 오류는 실제 OCR 문맥을 확인한 뒤 OCR 발생, JSON 구조화 발생,
표 빈칸 의미 규칙으로 분류한다. 자동 숫자 존재 검색만으로 관계를 단정하지 않기 위해 현재 발견된
오류 패턴별 명시적인 감사 규칙과 근거 문맥을 함께 저장한다.

In [1]:
from __future__ import annotations

import csv
import json
import os
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks/data/10_relational_critical_fact_evaluation/latest_run.json").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
V2_PATH = PROJECT_ROOT / "data/ocr_benchmark/gold/critical_rules/critical_rules_v2.json"
EVAL10_ROOT = PROJECT_ROOT / "notebooks/data/10_relational_critical_fact_evaluation"
OUTPUT_ROOT = PROJECT_ROOT / "notebooks/data/11_numeric_error_attribution"
RUN_ID = os.getenv("NUMERIC_ATTRIBUTION_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID

OCR_SOURCES = {
    "upstage_baseline": {
        "kind": "upstage",
        "root": PROJECT_ROOT / "data/ocr_benchmark/normalized/upstage",
    },
    "luna_original_repeat_1": {
        "kind": "page_text",
        "root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T143000Z_repeat01/ocr_text/api_luna_original",
    },
    "luna_original_repeat_2": {
        "kind": "page_text",
        "root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144500Z_api_repeat01/ocr_text/api_luna_original",
    },
    "terra_original_repeat_1": {
        "kind": "page_text",
        "root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T143000Z_repeat01/ocr_text/api_terra_original",
    },
    "terra_original_repeat_2": {
        "kind": "page_text",
        "root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144500Z_api_repeat01/ocr_text/api_terra_original",
    },
}


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0]) if rows else []
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        if fieldnames:
            writer.writeheader()
            writer.writerows(rows)


def read_upstage_page(root: Path, issuer: str, card_name: str, page_num: int) -> str:
    payload = read_json(root / issuer / f"{card_name}.json")
    page = next((item for item in payload.get("pages", []) if int(item["page_num"]) == page_num), None)
    if page is None:
        raise KeyError(f"Upstage 페이지 없음: {issuer}/{card_name} p{page_num}")
    parts = [block.get("text", "") for block in page.get("blocks", [])]
    parts += [table.get("content", "") for table in page.get("tables", [])]
    return "\n".join(part for part in parts if part)


def read_page_text(root: Path, issuer: str, card_name: str, page_num: int) -> str:
    text = (root / issuer / f"{card_name}.txt").read_text(encoding="utf-8")
    match = re.search(
        rf"^\[PAGE {page_num}\]\s*(.*?)(?=^\[PAGE \d+\]|\Z)",
        text,
        flags=re.MULTILINE | re.DOTALL,
    )
    if not match:
        raise KeyError(f"API OCR 페이지 없음: {issuer}/{card_name} p{page_num}")
    return match.group(1)


def ocr_page(run_name: str, issuer: str, card_name: str, page_num: int) -> str:
    source = OCR_SOURCES[run_name]
    if source["kind"] == "upstage":
        return read_upstage_page(source["root"], issuer, card_name, page_num)
    return read_page_text(source["root"], issuer, card_name, page_num)


def parse_cell(value: str) -> Any:
    return json.loads(value)


def compact_excerpt(text: str, anchors: list[str], radius: int = 520) -> str:
    compact = re.sub(r"[ \t]+", " ", text).strip()
    positions = [compact.casefold().find(anchor.casefold()) for anchor in anchors if anchor]
    positions = [position for position in positions if position >= 0]
    position = min(positions) if positions else 0
    start = max(0, position - 100)
    end = min(len(compact), position + radius)
    return compact[start:end].replace("\n", " ").strip()


ANCHORS = {
    "annual_fee": ["| 국내외겸용", "| 가족", "연회비"],
    "family_international_annual_fee": ["| 가족", "가족 | 해외겸용", "연회비"],
    "monthly_bill_discount": ["월납요금", "공과금 10%", "전월 실적"],
    "weekend_discount": ["주유/쇼핑", "주말 할인", "3대 할인마트"],
    "time_discount": ["Time 할인서비스", "Time할인서비스 할인한도"],
    "mileage_earning_unit": ["이용금액 1천원당", "1,000원으로 나누고"],
    "interest_free_installment": ["국내 2~3개월 무이자", "50만원 이상 2~3개월"],
    "minimum_installment_payment": ["국내 2~3개월 무이자", "50만원 이상 2~3개월"],
    "highest_spend_category_limit_low": ["전월 이용금액별 월 할인한도", "40만원 이상"],
    "highest_spend_category_limit_high": ["전월 이용금액별 월 할인한도", "70만원 이상"],
    "international_brand_fee_waiver": ["국제브랜드사 수수료", "1% 면제"],
    "overseas_service_fee_waiver": ["해외이용 수수료", "0.3% 면제"],
    "fuel_discount": ["할인한도", "월 30만원", "월 25만원"],
    "fuel_discount_monthly_spend_limit": ["할인한도", "월 30만원", "월 25만원"],
    "everland_discount": ["에버랜드할인", "에버랜드 페스티벌월드"],
}


latest10 = read_json(EVAL10_ROOT / "latest_run.json")
details10_path = PROJECT_ROOT / Path(latest10["summary"]).parent / "atomic_fact_details.csv"
with details10_path.open(encoding="utf-8", newline="") as handle:
    detail_rows = list(csv.DictReader(handle))

numeric_mismatches = [
    row for row in detail_rows
    if row["numeric_fact"] == "1"
    and row["prediction_supported"] == "1"
    and row["value_exact"] == "0"
]

v2 = read_json(V2_PATH)
facts = {
    (card["issuer"], card["card_name"], fact["fact_id"]): fact
    for card in v2["cards"] for fact in card["facts"]
}
print(f"귀속 대상 수치 불일치 라벨: {len(numeric_mismatches)}개")

귀속 대상 수치 불일치 라벨: 31개


In [2]:
def attribution_for(row: dict[str, str], fact: dict[str, Any]) -> dict[str, Any]:
    run_name = row["run_name"]
    issuer = row["issuer"]
    benefit = row["benefit_or_fee_id"]
    actual = parse_cell(row["actual_value"])

    if issuer == "hana" and benefit in {"interest_free_installment", "minimum_installment_payment"}:
        return {
            "attribution": "ocr_value_error",
            "ocr_gold_status": "incorrect_value",
            "ocr_observed_value": 500000,
            "json_status": "matches_incorrect_ocr",
            "json_matches_ocr": True,
            "root_issue_id": f"{run_name}:hana:installment_minimum_5_to_50_manwon",
            "reason": "gold는 5만원이지만 OCR이 50만원으로 읽었고 JSON도 500000으로 변환했다.",
            "confidence": "high",
        }

    if issuer == "kookmin" and benefit in {"fuel_discount", "fuel_discount_monthly_spend_limit"}:
        return {
            "attribution": "ocr_value_error",
            "ocr_gold_status": "incorrect_value",
            "ocr_observed_value": 300000,
            "json_status": "matches_incorrect_ocr",
            "json_matches_ocr": True,
            "root_issue_id": f"{run_name}:kookmin:fuel_monthly_limit_25_to_30_manwon",
            "reason": "gold는 월 25만원이지만 OCR이 월 30만원으로 읽었고 JSON도 300000으로 변환했다.",
            "confidence": "high",
        }

    if issuer == "kookmin" and benefit == "everland_discount":
        return {
            "attribution": "ocr_omission",
            "ocr_gold_status": "context_value_omitted",
            "ocr_observed_value": None,
            "json_status": "missing_after_ocr_omission",
            "json_matches_ocr": True,
            "root_issue_id": f"{run_name}:kookmin:everland_daily_limit_omitted",
            "reason": "gold의 에버랜드 '1일 1매'가 해당 OCR 문맥에서 빠졌고 JSON도 null을 반환했다.",
            "confidence": "high",
        }

    if actual is None and (
        (issuer == "samsung" and benefit == "family_international_annual_fee")
        or (issuer == "kookmin" and benefit == "annual_fee")
    ):
        return {
            "attribution": "table_blank_semantics",
            "ocr_gold_status": "implicit_zero_as_blank_or_dash",
            "ocr_observed_value": "blank_or_dash",
            "json_status": "null_instead_of_gold_zero",
            "json_matches_ocr": None,
            "root_issue_id": f"{run_name}:{issuer}:{benefit}:blank_means_zero",
            "reason": "OCR은 표의 빈칸 또는 '-'를 보존했다. gold는 이를 0원으로 해석했지만 JSON은 null로 두어 명시적 표 해석 규칙이 필요하다.",
            "confidence": "medium",
        }

    if run_name == "upstage_baseline" and issuer == "shinhan" and benefit in {
        "monthly_bill_discount", "weekend_discount", "time_discount"
    }:
        return {
            "attribution": "json_structure_omission",
            "ocr_gold_status": "correct_value_and_tier_table",
            "ocr_observed_value": parse_cell(row["expected_value"]),
            "json_status": "missing_despite_correct_ocr",
            "json_matches_ocr": False,
            "root_issue_id": f"{run_name}:shinhan:{benefit}:tier_table_omitted",
            "reason": "OCR에는 실적 구간과 할인한도 표가 정확히 있지만 JSON tiers 값이 null이다.",
            "confidence": "high",
        }

    if issuer == "samsung" and benefit in {
        "highest_spend_category_limit_low", "highest_spend_category_limit_high"
    }:
        return {
            "attribution": "json_relation_error",
            "ocr_gold_status": "correct_value_and_table_relation",
            "ocr_observed_value": parse_cell(row["expected_value"]),
            "json_status": "spend_threshold_bound_as_discount_limit",
            "json_matches_ocr": False,
            "root_issue_id": f"{run_name}:samsung:spend_threshold_bound_as_limit",
            "reason": "OCR 표는 40/70만원 실적과 5천/1만원 한도를 정확히 구분하지만 JSON이 실적값을 한도 필드에 넣었다.",
            "confidence": "high",
        }

    if issuer == "woori" and benefit in {
        "international_brand_fee_waiver", "overseas_service_fee_waiver"
    }:
        return {
            "attribution": "json_normalization_error",
            "ocr_gold_status": "correct_percentage",
            "ocr_observed_value": parse_cell(row["expected_value"]),
            "json_status": "percentage_not_scaled_to_ratio",
            "json_matches_ocr": False,
            "root_issue_id": f"{run_name}:woori:fee_percentage_ratio_normalization",
            "reason": "OCR은 1%와 0.3%를 정확히 읽었지만 JSON이 각각 1과 0.3으로 저장해 0.01/0.003 비율 정규화에 실패했다.",
            "confidence": "high",
        }

    if run_name == "upstage_baseline" and issuer == "woori" and benefit == "mileage_earning_unit":
        return {
            "attribution": "json_relation_error",
            "ocr_gold_status": "correct_spend_unit",
            "ocr_observed_value": 1000,
            "json_status": "mile_value_bound_as_spend_unit",
            "json_matches_ocr": False,
            "root_issue_id": f"{run_name}:woori:mileage_spend_unit_bound_to_mile",
            "reason": "OCR은 1천원당 1마일을 정확히 보존했지만 JSON이 지출 단위에 마일 값 1을 넣었다.",
            "confidence": "high",
        }

    raise AssertionError(
        "분류 규칙이 없는 수치 불일치: "
        f"{run_name} {issuer}/{row['card_name']} {benefit} "
        f"{row['expected_value']}->{row['actual_value']}"
    )


def normalized_evidence(text: str) -> str:
    return re.sub(r"[\s,|*#`]+", "", text).casefold()


def require_evidence(text: str, *terms: str) -> None:
    normalized = normalized_evidence(text)
    missing = [term for term in terms if normalized_evidence(term) not in normalized]
    if missing:
        raise AssertionError(f"OCR 근거 문구 누락: {missing}")


def require_table_row(text: str, pattern: str) -> None:
    if not re.search(pattern, text, flags=re.IGNORECASE):
        raise AssertionError(f"OCR 표 빈칸/- 근거 행 누락: {pattern}")


def validate_ocr_evidence(row: dict[str, str], decision: dict[str, Any], page_text: str) -> None:
    issuer = row["issuer"]
    benefit = row["benefit_or_fee_id"]
    run_name = row["run_name"]

    if issuer == "hana" and benefit in {"interest_free_installment", "minimum_installment_payment"}:
        require_evidence(page_text, "50만원 이상 2~3개월")
    elif issuer == "kookmin" and benefit in {"fuel_discount", "fuel_discount_monthly_spend_limit"}:
        require_evidence(page_text, "월 30만원 이내")
    elif issuer == "kookmin" and benefit == "everland_discount":
        normalized = normalized_evidence(page_text)
        start = normalized.find("에버랜드")
        end = normalized.find("캐리비안", start)
        if start < 0 or end < 0 or "1일1매" in normalized[start:end]:
            raise AssertionError("에버랜드 문맥의 1일 1매 누락 근거가 일치하지 않습니다.")
    elif run_name == "upstage_baseline" and issuer == "shinhan":
        if benefit in {"monthly_bill_discount", "weekend_discount"}:
            require_evidence(page_text, "3천원", "7천원", "1만원")
        elif benefit == "time_discount":
            require_evidence(page_text, "1만원", "2만원", "3만원")
    elif issuer == "samsung" and benefit in {"highest_spend_category_limit_low", "highest_spend_category_limit_high"}:
        require_evidence(page_text, "40만원 이상", "70만원 이상", "5,000원", "10,000원")
    elif issuer == "woori" and benefit in {"international_brand_fee_waiver", "overseas_service_fee_waiver"}:
        require_evidence(page_text, "국제브랜드사 수수료 1% 면제", "해외이용 수수료 0.3% 면제")
    elif run_name == "upstage_baseline" and issuer == "woori" and benefit == "mileage_earning_unit":
        require_evidence(page_text, "이용금액 1천원당", "1마일리지 적립")
    elif decision["attribution"] == "table_blank_semantics" and issuer == "samsung":
        require_table_row(page_text, r"\|\s*가족\s*\|\s*해외겸용\s*\|\s*Mastercard\s*\|\s*1만\s*3천원\s*\|\s*-?\s*\|\s*1만\s*3천원\s*\|")
    elif decision["attribution"] == "table_blank_semantics" and issuer == "kookmin":
        require_table_row(page_text, r"\|\s*국내외겸용\(비자\)\s*\|\s*골드\s*\|\s*10,?000원\s*\|\s*-?\s*\|\s*10,?000원\s*\|")


attribution_rows: list[dict[str, Any]] = []
for row in numeric_mismatches:
    scoped_key = (row["issuer"], row["card_name"], row["fact_id"])
    fact = facts[scoped_key]
    page_num = int(row["page_num"])
    page_text = ocr_page(row["run_name"], row["issuer"], row["card_name"], page_num)
    decision = attribution_for(row, fact)
    validate_ocr_evidence(row, decision, page_text)
    attribution_rows.append({
        "run_name": row["run_name"],
        "model_group": row["model_group"],
        "issuer": row["issuer"],
        "card_name": row["card_name"],
        "page_num": page_num,
        "fact_id": row["fact_id"],
        "benefit_or_fee_id": row["benefit_or_fee_id"],
        "metric": fact["metric"],
        "evaluation_role": row["evaluation_role"],
        "gold_value": row["expected_value"],
        "gold_unit": row["expected_unit"],
        "ocr_gold_status": decision["ocr_gold_status"],
        "ocr_observed_value": json.dumps(decision["ocr_observed_value"], ensure_ascii=False),
        "json_value": row["actual_value"],
        "json_status": decision["json_status"],
        "json_matches_ocr": "" if decision["json_matches_ocr"] is None else int(decision["json_matches_ocr"]),
        "attribution": decision["attribution"],
        "root_issue_id": decision["root_issue_id"],
        "confidence": decision["confidence"],
        "reason": decision["reason"],
        "ocr_evidence": compact_excerpt(page_text, ANCHORS.get(row["benefit_or_fee_id"], fact["source"].get("context_terms", []))),
    })

if len(attribution_rows) != len(numeric_mismatches):
    raise AssertionError("일부 수치 불일치가 귀속 결과에서 누락됐습니다.")
write_csv(RUN_ROOT / "numeric_error_attribution.csv", attribution_rows)
print(Counter(row["attribution"] for row in attribution_rows))

Counter({'json_structure_omission': 9, 'table_blank_semantics': 7, 'ocr_value_error': 6, 'json_normalization_error': 4, 'json_relation_error': 3, 'ocr_omission': 2})


In [3]:
root_issues: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in attribution_rows:
    root_issues[row["root_issue_id"]].append(row)

root_issue_rows = []
for issue_id, rows in sorted(root_issues.items()):
    first = rows[0]
    root_issue_rows.append({
        "root_issue_id": issue_id,
        "run_name": first["run_name"],
        "model_group": first["model_group"],
        "issuer": first["issuer"],
        "card_name": first["card_name"],
        "attribution": first["attribution"],
        "affected_label_rows": len(rows),
        "confidence": first["confidence"],
        "reason": first["reason"],
        "fact_ids": " | ".join(row["fact_id"] for row in rows),
    })
write_csv(RUN_ROOT / "root_issue_summary.csv", root_issue_rows)

by_run = []
for run_name in OCR_SOURCES:
    rows = [row for row in attribution_rows if row["run_name"] == run_name]
    issues = [row for row in root_issue_rows if row["run_name"] == run_name]
    row_counts = Counter(row["attribution"] for row in rows)
    issue_counts = Counter(row["attribution"] for row in issues)
    by_run.append({
        "run_name": run_name,
        "model_group": rows[0]["model_group"] if rows else "",
        "mismatched_numeric_label_rows": len(rows),
        "root_issue_occurrences": len(issues),
        "ocr_origin_label_rows": row_counts["ocr_value_error"] + row_counts["ocr_omission"],
        "json_origin_label_rows": row_counts["json_structure_omission"] + row_counts["json_relation_error"] + row_counts["json_normalization_error"],
        "table_blank_semantics_label_rows": row_counts["table_blank_semantics"],
        "ocr_origin_root_issues": issue_counts["ocr_value_error"] + issue_counts["ocr_omission"],
        "json_origin_root_issues": issue_counts["json_structure_omission"] + issue_counts["json_relation_error"] + issue_counts["json_normalization_error"],
        "table_blank_semantics_root_issues": issue_counts["table_blank_semantics"],
    })
write_csv(RUN_ROOT / "attribution_by_run.csv", by_run)

row_counts = Counter(row["attribution"] for row in attribution_rows)
issue_counts = Counter(row["attribution"] for row in root_issue_rows)
summary = {
    "schema_version": "numeric_error_attribution_v1",
    "run_id": RUN_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_evaluation": str(details10_path.relative_to(PROJECT_ROOT)),
    "numeric_mismatch_label_rows": len(attribution_rows),
    "root_issue_occurrences": len(root_issue_rows),
    "label_row_attribution": dict(sorted(row_counts.items())),
    "root_issue_attribution": dict(sorted(issue_counts.items())),
    "by_run": by_run,
    "interpretation": {
        "ocr_origin": "gold와 OCR 문맥이 다르고 JSON이 OCR의 오독 또는 누락을 그대로 따름",
        "json_origin": "OCR 문맥은 gold와 일치하지만 JSON 값, 경로 연결 또는 정규화가 틀림",
        "table_blank_semantics": "OCR은 빈칸/-를 보존했으며 gold의 0원 의미를 JSON에 적용할 명시 규칙이 없음",
    },
    "limitations": [
        "수기 gold와 structured gold가 정확하다는 전제의 귀속이다.",
        "현재 발견된 31개 수치 불일치에 대한 근거 기반 감사이며 새로운 오류 패턴은 명시 규칙을 추가해야 한다.",
        "semantic 관계와 numeric probe가 같은 원인을 중복 표시할 수 있어 label row와 root issue occurrence를 함께 제공한다.",
        "표 빈칸을 0으로 해석하는 규칙은 OCR 정확도 문제가 아니라 gold/구조화 스키마의 표현 규칙 문제다.",
    ],
}
write_json(RUN_ROOT / "summary.json", summary)
write_json(OUTPUT_ROOT / "latest_run.json", {
    "latest_run_id": RUN_ID,
    "summary": str((RUN_ROOT / "summary.json").relative_to(PROJECT_ROOT)),
})

print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"결과 저장: {RUN_ROOT.relative_to(PROJECT_ROOT)}")

{
  "schema_version": "numeric_error_attribution_v1",
  "run_id": "20260810T060240Z",
  "generated_at": "2026-08-10T06:20:25.999953+00:00",
  "source_evaluation": "notebooks/data/10_relational_critical_fact_evaluation/runs/20260808T082345Z/atomic_fact_details.csv",
  "numeric_mismatch_label_rows": 31,
  "root_issue_occurrences": 19,
  "label_row_attribution": {
    "json_normalization_error": 4,
    "json_relation_error": 3,
    "json_structure_omission": 9,
    "ocr_omission": 2,
    "ocr_value_error": 6,
    "table_blank_semantics": 7
  },
  "root_issue_attribution": {
    "json_normalization_error": 2,
    "json_relation_error": 2,
    "json_structure_omission": 3,
    "ocr_omission": 2,
    "ocr_value_error": 3,
    "table_blank_semantics": 7
  },
  "by_run": [
    {
      "run_name": "upstage_baseline",
      "model_group": "Upstage Document Parse",
      "mismatched_numeric_label_rows": 12,
      "root_issue_occurrences": 6,
      "ocr_origin_label_rows": 0,
      "json_origin_la

## 판정 해석

- `ocr_value_error`: OCR이 정답과 다른 수치를 만들었고 JSON이 그 값을 따라갔다.
- `ocr_omission`: OCR에서 조건 수치가 빠졌고 JSON도 값을 만들지 못했다.
- `json_structure_omission`: OCR에는 정답 표가 있지만 JSON이 해당 필드를 누락했다.
- `json_relation_error`: OCR 값은 맞지만 JSON이 다른 행·열의 값을 연결했다.
- `json_normalization_error`: OCR의 `%` 표기는 맞지만 JSON 비율 값으로 정규화하지 못했다.
- `table_blank_semantics`: 표 빈칸/`-`를 gold에서는 0으로 해석했지만 JSON은 null로 유지했다.